Delete Duplicate
Schema Evolution
DataTypes Handleing / Type casting
Null Records Handling
Triming
Case Conversion
Maintain Order of Data

In [0]:
#Dropdown widget for environment selection
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="dev",
    choices=["dev", "prd", "qa"],
    label="select Environment"
)
#Get selected environment
env = dbutils.widgets.get("environment")

#Define dynamic table names and source file location
silverTablName = f"saleslake_{env}.silver_{env}.cleanedCustomer"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawcustomer"
srcFileLoc     = f"/Volumes/saleslake_{env}/silver_{env}/vol_saleslake_src_files_{env}/dailyCustomer/"

In [0]:
#Insert into Silver with cleansing rules
spark.sql(f"""
INSERT INTO {silverTablName} 
SELECT DISTINCT
CAST(TRIM(customer_id)AS INT)as customer_id,
UPPER(TRIM(customer_name)) as customer_name,
UPPER(TRIM(email))as email,
TRIM(phone)as phone,
UPPER(TRIM(address))as address, 
UPPER(TRIM(city))as city,
UPPER(TRIM(state))as state,
UPPER(TRIM(country))as country, 
UPPER(TRIM(zip_code))as zip_code,
UPPER(TRIM(segment))as segment,
CURRENT_TIMESTAMP() as ingest_ts
FROM {bronzeTablName} 
WHERE ingest_ts > (
                     SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                     FROM {silverTablName} 
                    )
 ORDER BY CAST(TRIM(customer_id) AS INT)""")
